# AI agents

Agents for exploring data, creating budgets, and what-if scenarios.
What-if uses a two-phase architecture: deterministic compute + LLM narration only.

In [ ]:
#| default_exp agents

In [ ]:
#| export
import os
import re
import sys
import json
import difflib
import calendar
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
from typing import Optional, Dict, Any, List, Tuple
import pandas as pd

In [ ]:
#| export
def _llm_completion(messages: List[Dict[str, str]], max_tokens: int = 500) -> Optional[str]:
    """Call OpenAI chat completion. Returns None if key missing or error."""
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return None
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        r = client.chat.completions.create(
            model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
            messages=messages,
            max_tokens=max_tokens,
        )
        return r.choices[0].message.content if r.choices else None
    except Exception as e:
        print(f"[agents] LLM error: {e}", file=sys.stderr)
        return None

In [ ]:
#| export
def parse_date_range(msg: str) -> Tuple[date, date, str]:
    """
    Parse natural language time references into (start_date, end_date, label).
    Handles: month names, 'last N months', 'this month', 'last month', 'this year', etc.
    """
    today = date.today()
    m = msg.strip().lower()

    month_names = {name.lower(): num for num, name in enumerate(calendar.month_name) if num}
    month_abbr = {name.lower(): num for num, name in enumerate(calendar.month_abbr) if num}
    all_months = {**month_names, **month_abbr}

    for name, num in all_months.items():
        if name in m:
            year_match = re.search(rf'{name}\s*(\d{{4}})', m)
            year = int(year_match.group(1)) if year_match else today.year
            if num > today.month and year == today.year:
                year -= 1
            start = date(year, num, 1)
            end = (start + relativedelta(months=1)) - timedelta(days=1)
            label = start.strftime("%B %Y")
            return start, min(end, today), label

    n_match = re.search(r'last\s+(\d+)\s+month', m)
    if n_match:
        n = int(n_match.group(1))
        start = today - relativedelta(months=n)
        return start, today, f"Last {n} months"

    if "last month" in m or "previous month" in m:
        start = (today.replace(day=1) - timedelta(days=1)).replace(day=1)
        end = today.replace(day=1) - timedelta(days=1)
        return start, end, start.strftime("%B %Y")

    if "this year" in m:
        return date(today.year, 1, 1), today, str(today.year)

    if "last year" in m:
        return date(today.year - 1, 1, 1), date(today.year - 1, 12, 31), str(today.year - 1)

    start = today.replace(day=1)
    return start, today, start.strftime("%B %Y")

In [ ]:
#| export
def _run_read_only_query(template_id: str, params: Optional[Dict[str, Any]] = None) -> pd.DataFrame:
    """Run a safe read-only query template. No raw SQL from caller."""
    from jupyter_finance.core import db_sql, SQL_ENGINE
    from sqlalchemy import create_engine, text

    params = params or {}
    templates = {
        "spending_by_category": """
            SELECT COALESCE(user_friendly_category, personal_finance_category) AS category,
                   SUM(amount) AS total
            FROM transactions
            WHERE date >= :start_date AND date <= :end_date AND amount > 0
            GROUP BY 1 ORDER BY total DESC
        """,
        "budgets_over_limit": """
            SELECT b.name, b.balance_limit, bb.current_balance, bb.under_limit
            FROM budget b
            JOIN v_latest_budget_batches bb ON bb.budget_id = b.id
            WHERE b.is_deleted = false AND bb.under_limit = false
        """,
        "budget_summary": """
            SELECT b.id, b.name, b.balance_limit, bb.current_balance, bb.under_limit
            FROM budget b
            LEFT JOIN v_latest_budget_batches bb ON bb.budget_id = b.id
            WHERE b.is_deleted = false
        """,
        "last_month_total": """
            SELECT SUM(amount) AS total
            FROM transactions
            WHERE date >= date_trunc('month', CURRENT_DATE - interval '1 month')::date
              AND date < date_trunc('month', CURRENT_DATE)::date AND amount > 0
        """,
        "top_merchants": """
            SELECT merchant_name, SUM(amount) AS total, COUNT(*) AS cnt
            FROM transactions
            WHERE date >= :start_date AND date <= :end_date AND amount > 0
            GROUP BY merchant_name ORDER BY total DESC LIMIT 20
        """,
        "budget_batch_history": """
            SELECT b.name, bb.start_date, bb.end_date, bb.current_balance, b.balance_limit, bb.under_limit
            FROM budget b
            JOIN budget_batch bb ON bb.budget_id = b.id
            WHERE b.is_deleted = false
            ORDER BY b.name, bb.start_date
        """,
        "budget_transactions": """
            SELECT b.name AS budget_name, t.date, t.merchant_name, t.amount,
                   COALESCE(t.user_friendly_category, t.personal_finance_category) AS category
            FROM budgeted_transaction bt
            JOIN budget_batch bb ON bt.batch_id = bb.id
            JOIN budget b ON bb.budget_id = b.id
            JOIN transactions t ON bt.transaction_id = t.transaction_id
            WHERE b.is_deleted = false AND bb.start_date >= :start_date AND bb.end_date <= :end_date
            ORDER BY t.date DESC
        """,
    }
    sql = templates.get(template_id)
    if not sql:
        return pd.DataFrame()
    try:
        if params:
            engine = create_engine(SQL_ENGINE)
            df = pd.read_sql_query(text(sql), engine, params=params)
            engine.dispose()
            return df
        return db_sql(sql)
    except Exception:
        return pd.DataFrame()

In [ ]:
#| export
_CATEGORY_ALIASES = {
    "eating out": ["FOOD_AND_DRINK_RESTAURANTS", "Discretionary"],
    "restaurants": ["FOOD_AND_DRINK_RESTAURANTS", "Discretionary"],
    "dining": ["FOOD_AND_DRINK_RESTAURANTS", "Discretionary"],
    "dining out": ["FOOD_AND_DRINK_RESTAURANTS", "Discretionary"],
    "groceries": ["FOOD_AND_DRINK_GROCERIES", "Essentials"],
    "coffee": ["FOOD_AND_DRINK_COFFEE", "Discretionary"],
    "rent": ["RENT_AND_UTILITIES", "Essentials"],
    "utilities": ["RENT_AND_UTILITIES", "Essentials"],
    "shopping": ["SHOPPING", "Discretionary"],
    "entertainment": ["ENTERTAINMENT", "Discretionary"],
    "travel": ["TRAVEL", "Discretionary"],
    "subscriptions": ["ENTERTAINMENT_TV_AND_MOVIES", "Discretionary"],
    "gym": ["RECREATION_GYMS_AND_FITNESS", "Discretionary"],
    "transport": ["TRANSPORTATION", "Discretionary"],
    "healthcare": ["HEALTHCARE", "Essentials"],
    "insurance": ["INSURANCE", "Essentials"],
}


def match_category(user_term: str, available_cats: List[str]) -> Optional[str]:
    """
    Fuzzy-match a user-provided category name against available categories.
    Three-pass strategy: aliases -> exact case-insensitive -> difflib fuzzy -> substring.
    """
    term = user_term.strip().lower()
    if not term or not available_cats:
        return None
    lower_map = {c.lower(): c for c in available_cats}

    if term in _CATEGORY_ALIASES:
        for alias in _CATEGORY_ALIASES[term]:
            if alias.lower() in lower_map:
                return lower_map[alias.lower()]

    if term in lower_map:
        return lower_map[term]

    matches = difflib.get_close_matches(term, list(lower_map.keys()), n=1, cutoff=0.5)
    if matches:
        return lower_map[matches[0]]

    for low, orig in lower_map.items():
        if term in low or low in term:
            return orig

    return None

In [ ]:
#| export
def _parse_whatif_intent(msg: str, available_categories: List[str]) -> Optional[List[Dict]]:
    """Use LLM to extract structured what-if rules from natural language."""
    cats_str = ", ".join(available_categories[:20])
    prompt = f"""Extract the what-if spending rule(s) from the user's message.

Available spending categories: {cats_str}

Return ONLY valid JSON. Schema:
{{"rules": [{{"category": "name", "action": "cut|zero_out|cap", "pct": number_or_null, "cap_amount": number_or_null}}]}}

Action mapping:
- "spend no more", "eliminate", "stop", "zero" -> action="zero_out", pct=100
- "cut by X%", "reduce by X%" -> action="cut", pct=X
- "limit to $X", "cap at $X" -> action="cap", cap_amount=X

User message: {msg}"""

    out = _llm_completion([
        {"role": "system", "content": "Return only valid JSON matching the schema. No markdown, no explanation."},
        {"role": "user", "content": prompt}
    ], max_tokens=200)
    if not out:
        return None
    try:
        s = out.strip()
        if s.startswith("```"):
            s = s.split("```")[1]
            if s.startswith("json"):
                s = s[4:]
        data = json.loads(s)
        rules = data.get("rules", [])
        if not isinstance(rules, list) or not rules:
            return None
        valid = []
        for r in rules:
            action = r.get("action", "cut")
            if action not in ("cut", "zero_out", "cap"):
                action = "cut"
            valid.append({
                "category": str(r.get("category", "")),
                "action": action,
                "pct": float(r["pct"]) if r.get("pct") is not None else None,
                "cap_amount": float(r["cap_amount"]) if r.get("cap_amount") is not None else None,
            })
        return valid or None
    except (json.JSONDecodeError, ValueError, KeyError, TypeError) as e:
        print(f"[whatif_agent] Intent parse error: {e}", file=sys.stderr)
        return None

In [ ]:
#| export
def _apply_whatif_rules(
    spending: pd.DataFrame,
    cat_col: Optional[str],
    rules: List[Dict],
    start: date,
    end: date,
) -> Dict[str, Any]:
    """Deterministic what-if computation. No LLM calls."""
    today = date.today()
    is_mid_period = start <= today <= end
    cat_totals = spending.groupby(cat_col)["amount"].sum().to_dict() if cat_col else {}
    original_total = spending["amount"].sum()

    days_elapsed = max((min(today, end) - start).days, 1)
    days_total = max((end - start).days, 1)
    days_remaining = max((end - today).days, 0) if is_mid_period else 0

    if is_mid_period and not spending.empty:
        actual_mask = spending["date"].dt.date <= today
        actual_df = spending[actual_mask]
        spent_so_far = actual_df["amount"].sum()
        daily_rate = spent_so_far / days_elapsed
        projected_remaining = daily_rate * days_remaining
        projected_total = spent_so_far + projected_remaining
        cat_actual = actual_df.groupby(cat_col)["amount"].sum().to_dict() if cat_col else {}
        cat_proj_remaining = {}
        for cat, total in cat_totals.items():
            cs = cat_actual.get(cat, 0)
            cat_proj_remaining[cat] = (cs / days_elapsed) * days_remaining if days_elapsed else 0
    else:
        spent_so_far = original_total
        projected_remaining = 0.0
        projected_total = original_total
        cat_actual = dict(cat_totals)
        cat_proj_remaining = {k: 0.0 for k in cat_totals}

    per_rule = []
    seen = set()
    total_savings = 0.0
    unmatched = []

    for rule in rules:
        cat_name = rule.get("matched_to") or rule.get("category", "")
        if not cat_name or cat_name not in cat_totals:
            unmatched.append(rule.get("category", ""))
            continue
        if cat_name in seen:
            continue
        seen.add(cat_name)
        cat_orig = cat_totals[cat_name]
        action = rule.get("action", "cut")

        if is_mid_period:
            remaining = cat_proj_remaining.get(cat_name, 0)
            actual_spent = cat_actual.get(cat_name, 0)
            if action == "zero_out":
                adj_rem = 0.0
            elif action == "cut" and rule.get("pct"):
                adj_rem = remaining * (1 - rule["pct"] / 100)
            elif action == "cap" and rule.get("cap_amount") is not None:
                adj_rem = max(0, rule["cap_amount"] - actual_spent)
            else:
                adj_rem = remaining
            savings = remaining - adj_rem
            adjusted = actual_spent + adj_rem
        else:
            if action == "zero_out":
                adjusted = 0.0
            elif action == "cut" and rule.get("pct"):
                adjusted = cat_orig * (1 - rule["pct"] / 100)
            elif action == "cap" and rule.get("cap_amount") is not None:
                adjusted = min(cat_orig, rule["cap_amount"])
            else:
                adjusted = cat_orig
            savings = cat_orig - adjusted

        total_savings += savings
        per_rule.append({
            "category": cat_name, "action": action,
            "pct": rule.get("pct"), "cap_amount": rule.get("cap_amount"),
            "original": round(cat_orig, 2), "adjusted": round(adjusted, 2),
            "savings": round(savings, 2),
        })

    all_categories = {}
    for cat, total in cat_totals.items():
        adj = total
        for pr in per_rule:
            if pr["category"] == cat:
                adj = pr["adjusted"]
                break
        all_categories[cat] = {"original": round(total, 2), "adjusted": round(adj, 2)}

    new_total = projected_total - total_savings if is_mid_period else original_total - total_savings

    daily_spending = pd.DataFrame()
    if is_mid_period and not spending.empty:
        grp = (
            spending[spending["date"].dt.date <= today]
            .groupby(spending["date"].dt.date)["amount"].sum()
            .sort_index().cumsum().reset_index()
        )
        grp.columns = ["date", "cumulative"]
        daily_spending = grp

    return {
        "start": start, "end": end, "is_mid_period": is_mid_period,
        "original_total": round(original_total, 2),
        "projected_total": round(projected_total, 2),
        "new_total": round(new_total, 2),
        "total_savings": round(total_savings, 2),
        "per_rule": per_rule, "all_categories": all_categories,
        "spent_so_far": round(spent_so_far, 2),
        "projected_remaining": round(projected_remaining, 2),
        "days_elapsed": days_elapsed, "days_remaining": days_remaining,
        "daily_spending": daily_spending,
        "unmatched_categories": unmatched,
        "cat_actual": {k: round(v, 2) for k, v in cat_actual.items()},
    }

In [ ]:
#| export
def _build_narration(msg: str, label: str, start: date, end: date,
                     result: Dict, unmatched: List[str]) -> str:
    """Build narration from pre-computed numbers. LLM adds natural language only."""
    per_rule_text = ""
    for pr in result["per_rule"]:
        desc = {"cut": f"cut by {pr['pct']:.0f}%" if pr.get("pct") else "adjusted",
                "zero_out": "eliminated",
                "cap": f"capped at ${pr['cap_amount']:,.2f}" if pr.get("cap_amount") else "adjusted",
                }.get(pr["action"], "adjusted")
        per_rule_text += (f"- {pr['category']}: {desc}. "
                         f"Was ${pr['original']:,.2f}, now ${pr['adjusted']:,.2f} "
                         f"(saves ${pr['savings']:,.2f})\n")

    unmatched_text = ""
    if unmatched:
        avail = ", ".join(result["all_categories"].keys())
        unmatched_text = f"\nCould not match: {', '.join(unmatched)}. Available: {avail}."

    prorate_text = ""
    if result["is_mid_period"]:
        prorate_text = (f"\nMid-period projection. Spent so far: ${result['spent_so_far']:,.2f}. "
                       f"Projected remaining: ${result['projected_remaining']:,.2f} "
                       f"over {result['days_remaining']} days.")

    context = f"""User asked: {msg}
Period: {label} ({start} to {end})
{prorate_text}

Pre-computed results (use these EXACT numbers):
Original total: ${result['original_total']:,.2f}
{per_rule_text}Total savings: ${result['total_savings']:,.2f}
New projected total: ${result['new_total']:,.2f}
{unmatched_text}

Write 2-4 sentences summarizing these results in plain English.
Do NOT use any markdown formatting: no asterisks, no backticks, no code blocks, no bold.
Just write plain sentences with dollar amounts. Do NOT recalculate anything."""

    try:
        reply = _llm_completion([
            {"role": "system", "content": "Summarize financial what-if results using the exact pre-computed numbers. Be concise. Write plain text only, no markdown, no asterisks, no bold."},
            {"role": "user", "content": context}
        ], max_tokens=250)
    except Exception as e:
        print(f"[whatif_agent] Narration LLM error: {e}", file=sys.stderr)
        reply = None

    if reply:
        reply = re.sub(r'```[\s\S]*?```', '', reply)
        reply = re.sub(r'`([^`]*)`', r'\1', reply)
        reply = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', reply)
        reply = re.sub(r'#{1,3}\s*', '', reply)
        return reply.strip()

    parts = [f"For {label} ({start} to {end}):"]
    if result["is_mid_period"]:
        parts.append(f"You've spent ${result['spent_so_far']:,.2f} so far "
                     f"with {result['days_remaining']} days remaining.")
    for pr in result["per_rule"]:
        verb = "Eliminating" if pr["action"] == "zero_out" else "Cutting"
        parts.append(f"{verb} {pr['category']} saves ${pr['savings']:,.2f}.")
    parts.append(f"New projected total: ${result['new_total']:,.2f} "
                 f"(saving ${result['total_savings']:,.2f}).")
    if unmatched:
        parts.append(f"Could not match: {', '.join(unmatched)}.")
    return " ".join(parts)

In [ ]:
#| export
def whatif_compute(user_message: str) -> Dict[str, Any]:
    """
    Full what-if orchestrator. Returns {narration: str, result: dict|None}.
    The result dict contains all pre-computed figures for visualization.
    """
    msg = (user_message or "").strip()
    if not msg:
        return {"narration": "Ask e.g. 'What if I cut discretionary by 20%?' "
                "or 'For March, what if I spend no more on groceries?'", "result": None}

    try:
        from jupyter_finance.categorization import get_transactions_enriched
        df = get_transactions_enriched()
    except Exception:
        return {"narration": "No transaction data available for what-if analysis.", "result": None}
    if df.empty or "amount" not in df.columns:
        return {"narration": "No transaction data available.", "result": None}

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    start, end, label = parse_date_range(msg)
    period_df = df[(df["date"].dt.date >= start) & (df["date"].dt.date <= end)]

    cat_col = "user_friendly_category" if "user_friendly_category" in period_df.columns else "personal_finance_category"
    if cat_col not in period_df.columns:
        cat_col = None

    spending = period_df[period_df["amount"] > 0].copy()
    if spending.empty:
        return {"narration": f"No spending data for {label}.", "result": None}

    available_cats = sorted(spending[cat_col].dropna().unique().tolist()) if cat_col else []
    sub_cats = []
    if "personal_finance_category_detailed" in spending.columns:
        sub_cats = sorted(spending["personal_finance_category_detailed"].dropna().unique().tolist())

    rules = _parse_whatif_intent(msg, available_cats + sub_cats)

    if not rules:
        cat_totals = spending.groupby(cat_col)["amount"].sum().to_dict() if cat_col else {}
        total = spending["amount"].sum()
        lines = [f"  {k}: ${v:,.2f}" for k, v in sorted(cat_totals.items(), key=lambda x: -x[1])]
        fallback = (f"Spending for **{label}** ({start} to {end}): ${total:,.2f}\n"
                   f"By category:\n" + "\n".join(lines) +
                   f"\n\nTry: 'What if I cut Discretionary by 20% in {label}?'")
        return {"narration": fallback, "result": None}

    unmatched = []
    for rule in rules:
        raw = rule["category"]
        matched = match_category(raw, available_cats)
        if not matched and sub_cats:
            matched = match_category(raw, sub_cats)
            if matched and cat_col == "user_friendly_category":
                parent = spending[spending["personal_finance_category_detailed"] == matched]
                if not parent.empty:
                    matched = parent[cat_col].iloc[0]
        rule["matched_to"] = matched
        if not matched:
            unmatched.append(raw)

    result = _apply_whatif_rules(spending, cat_col, rules, start, end)
    result["label"] = label
    narration = _build_narration(msg, label, start, end, result, unmatched)
    return {"narration": narration, "result": result}

In [ ]:
#| export
def whatif_agent(user_message: str) -> str:
    """Thin wrapper: returns narration text only (for backward compat)."""
    return whatif_compute(user_message)["narration"]

In [ ]:
#| export
def explore_agent(user_message: str) -> str:
    """
    Answer natural-language questions about transactions and budgets.
    All responses include the date range being queried.
    """
    from jupyter_finance.core import get_all_active_budgets

    msg_lower = (user_message or "").strip().lower()
    if not msg_lower:
        return "Ask a question about your spending or budgets."

    start, end, label = parse_date_range(msg_lower)
    params = {"start_date": start.isoformat(), "end_date": end.isoformat()}

    if "over limit" in msg_lower or "over budget" in msg_lower:
        df = _run_read_only_query("budgets_over_limit")
        if df.empty:
            return f"No budgets are over their limit (as of {label})."
        return f"Budgets over limit ({label}):\n" + df.to_string(index=False)

    if "budget" in msg_lower and ("list" in msg_lower or "summary" in msg_lower or "all" in msg_lower):
        df = _run_read_only_query("budget_summary")
        if df.empty:
            return "No active budgets."
        return f"Budget summary (latest batch):\n" + df.to_string(index=False)

    if "biggest" in msg_lower or "largest" in msg_lower or "top" in msg_lower or "most" in msg_lower:
        df = _run_read_only_query("spending_by_category", params)
        if df.empty:
            return f"No spending data for {label}."
        lines = [f"  {r['category']}: ${r['total']:,.2f}" for _, r in df.head(8).iterrows()]
        return f"Your biggest expenses for **{label}** ({start} to {end}):\n" + "\n".join(lines)

    if "category" in msg_lower or "categories" in msg_lower or "spending by" in msg_lower:
        df = _run_read_only_query("spending_by_category", params)
        if df.empty:
            return f"No spending data for {label}."
        lines = [f"  {r['category']}: ${r['total']:,.2f}" for _, r in df.iterrows()]
        return f"Spending by category for **{label}** ({start} to {end}):\n" + "\n".join(lines)

    if "merchant" in msg_lower:
        df = _run_read_only_query("top_merchants", params)
        if df.empty:
            return f"No merchant data for {label}."
        return f"Top merchants for **{label}** ({start} to {end}):\n" + df.head(15).to_string(index=False)

    try:
        from jupyter_finance.categorization import get_transactions_enriched
        all_df = get_transactions_enriched()
        all_df["date"] = pd.to_datetime(all_df["date"], errors="coerce")
        period_df = all_df[(all_df["date"].dt.date >= start) & (all_df["date"].dt.date <= end)]
        sample = period_df.head(30).to_string() if len(period_df) else "No transactions."
    except Exception:
        sample = "No transactions."
    budgets_df = get_all_active_budgets()
    budget_sample = budgets_df.head(5).to_string() if not budgets_df.empty else "No budgets."
    prompt = f"""The user asked: {user_message}
Date range: {start} to {end} ({label})

Available data summary:
- Budgets: {budget_sample}
- Transactions in period (sample): {sample}

Reply in 1-3 short sentences. Always mention the date range. Use the numbers above. Do not make up numbers. Do NOT use backticks or code blocks."""
    out = _llm_completion([{"role": "user", "content": prompt}])
    return out or f"I couldn't answer that for {label}. Try: 'Spending by category for March' or 'Budgets over limit'."

In [ ]:
#| export
def create_budget_agent(user_message: str) -> str:
    """Parse user intent to create a budget and call insert_new_budget."""
    from jupyter_finance.core import insert_new_budget

    msg = (user_message or "").strip()
    if not msg:
        return "Describe the budget you want: name, spending limit, and cadence (weekly/biweekly/monthly)."

    prompt = f"""The user wants to create a budget. Extract from their message:
- name: short budget name (e.g. Groceries, Coffee)
- limit: number only, the spending limit in dollars
- cadence: one of weekly, biweekly, monthly
- day: one of mon, tue, wed, thu, fri, sat, sun (refresh day)
- rules: SQL WHERE clause for transactions, using columns: personal_finance_category, personal_finance_category_detailed, merchant_name. If unclear use NULL.

User message: {msg}

Reply with a single JSON object only, no markdown, keys: name, limit, cadence, day, rules. Use null for missing."""
    out = _llm_completion([{"role": "user", "content": prompt}], max_tokens=300)
    if not out:
        return "Could not parse your request. Say e.g. 'Create a monthly Coffee budget of 50 dollars'."
    try:
        s = out.strip()
        if s.startswith("```"):
            s = s.split("```")[1]
            if s.startswith("json"):
                s = s[4:]
        data = json.loads(s)
        name = (data.get("name") or "Unnamed").strip()[:255]
        limit = float(data.get("limit") or 0)
        cadence = (data.get("cadence") or "monthly").lower()
        day = (data.get("day") or "sun").lower()
        rules = data.get("rules")
        if not name or limit <= 0:
            return "Could not extract budget name and limit. Please specify name and a positive limit."
        if cadence not in ("weekly", "biweekly", "monthly"):
            cadence = "monthly"
        if day not in ("mon", "tue", "wed", "thu", "fri", "sat", "sun"):
            day = "sun"
        insert_new_budget(name=name, description=f"Created from: {msg[:200]}", limit=limit, cadence=cadence, date_of_week=day, rules=rules)
        return f"Budget '{name}' created with limit ${limit:.2f} ({cadence}, refresh {day})."
    except (json.JSONDecodeError, ValueError) as e:
        return f"Could not create budget: {e}. Try: 'Create a monthly Groceries budget of 400 dollars'."